In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF')

if HF_TOKEN:
    login(HF_TOKEN)
    print("Successfully logged in to Hugging Face!")
else:
    print("Token is not set. Please save the token first.")

In [ ]:
import torch
from transformers import AutoTokenizer
from pathlib import Path

from subnet_model import SubnetLLM

In [ ]:
# Configuration
MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
CACHE_DIR = './cache'
CHECKPOINT_DIR = Path('/content/drive/MyDrive/YORO/checkpoints')
CHECKPOINT_NAME = 'checkpoint_latest.pt'

DEVICE = 'cuda'
DTYPE = torch.bfloat16

# Number of tokens to generate
N_TOKENS = 50

In [ ]:
assert torch.cuda.is_available(), "CUDA is not available"

# Load checkpoint to get model config
checkpoint_path = CHECKPOINT_DIR / CHECKPOINT_NAME
print(f"Loading checkpoint from {checkpoint_path}...")
checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

config = checkpoint['config']
print(f"Model config: {config}")

In [ ]:
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Initialize model with config from checkpoint
print("Initializing model...")
model = SubnetLLM(
    model_name=config['model_name'],
    cache_dir=CACHE_DIR,
    embedding_layers=config['embedding_layers'],
    coherence_layers=config['coherence_layers'],
    adaptation_layers=config['adaptation_layers'],
    compensation_layers=config['compensation_layers'],
    concatenation_layers=config['concatenation_layers'],
    device=DEVICE,
    dtype=DTYPE
)

# Load model weights
model_state_dict = checkpoint['model_state_dict']

# Handle compiled model state dicts (strip _orig_mod. prefix if present)
if any(key.startswith('_orig_mod.') for key in model_state_dict.keys()):
    print("Detected compiled model checkpoint, stripping _orig_mod. prefix...")
    model_state_dict = {
        key.replace('_orig_mod.', ''): value
        for key, value in model_state_dict.items()
    }

model.load_state_dict(model_state_dict)
model.eval()
print("Model loaded successfully!")

In [ ]:
def generate(prompt: str, max_tokens: int = N_TOKENS) -> str:
    """
    Generate tokens autoregressively using greedy decoding.

    Args:
        prompt: The input prompt string
        max_tokens: Maximum number of tokens to generate

    Returns:
        The generated text including the prompt
    """
    # Tokenize prompt
    prompt_ids = tokenizer(prompt, return_tensors="pt")['input_ids'].to(DEVICE)
    attention_mask = torch.ones_like(prompt_ids)

    # Initialize cache
    cached_reasoning_outputs = [None]
    generated = prompt_ids

    with torch.no_grad():
        for _ in range(max_tokens):
            # Forward pass
            logits, cached_reasoning_outputs = model(
                input_ids=generated,
                cached_reasoning_outputs=cached_reasoning_outputs,
                attention_mask=attention_mask,
                use_teacher_forcing=False
            )

            # Greedy decoding: select token with highest probability
            next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)

            # Append to generated sequence
            generated = torch.cat([generated, next_token], dim=1)
            attention_mask = torch.ones_like(generated)

            # Stop if EOS token is generated
            if next_token.item() == tokenizer.eos_token_id:
                break

    return tokenizer.decode(generated[0][prompt_ids.shape[1]:].tolist(), skip_special_tokens=True)

In [ ]:
def generate_chat(messages: list, max_tokens: int = N_TOKENS) -> str:
    """
    Generate a response for a chat conversation.

    Args:
        messages: List of message dicts with 'role' and 'content' keys
        max_tokens: Maximum number of tokens to generate

    Returns:
        The generated response text
    """
    # Apply chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    return generate(prompt, max_tokens)

In [ ]:
# Example: Chat conversation
messages = [
    {'role': 'system', 'content': 'You are a helpful assistant.'},
    {'role': 'user', 'content': '1+1=?'}
]

print("Chat conversation:")
for msg in messages:
    print(f"  {msg['role']}: {msg['content']}")
print(f"\nGenerated response:\n{generate_chat(messages)}")

In [ ]:
from typing import Any

DEFAULT_SYSTEM_PROMPT = 'You are a helpful assistant. You will be provided a request. Satisfy the request.'

def parse_alpaca_gpt4(row: dict[str, Any]) -> dict[str, str]:
  parts = row['text'].split('### Instruction:')
  return {
    'system': parts[0].strip(),
    'user': parts[1].split('### Response:')[0].strip()
  }

def parse_databricks_dolly_15k(row: dict[str, Any]) -> dict[str, str]:
  has_context = len(row['context'].strip()) > 0
  return {
    'system': 'You are a helpful assistant. You will be provided a context and a request. Satisfy the request, given the context.' if has_context else DEFAULT_SYSTEM_PROMPT,
    'user': f'### Context:\n{row["context"]}\n\n###Request:\n{row["instruction"]}' if has_context else f'###Request:\n{row["instruction"]}'
  }

def parse_wizardlm_evol_instruct_v2_196k(row: dict[str, Any]) -> dict[str, str]:
  return {
    'system': DEFAULT_SYSTEM_PROMPT,
    'user': row['conversations'][0]['value']
  }

def parse_slimorca(row: dict[str, Any]) -> dict[str, str]:
  return {
    'system': row['conversations'][0]['value'] if row['conversations'][0]['from'] == 'system' else DEFAULT_SYSTEM_PROMPT,
    'user': row['conversations'][1]['value'] if row['conversations'][0]['from'] == 'system' else row['conversations'][0]['value']
  }

def parse_openhermes_25(row: dict[str, Any]) -> dict[str, str]:
  return {
    'system': row['conversations'][0]['value'] if row['conversations'][0]['from'] == 'system' else DEFAULT_SYSTEM_PROMPT,
    'user': row['conversations'][1]['value'] if row['conversations'][0]['from'] == 'system' else row['conversations'][0]['value']
  }


In [ ]:
row = {
    'text': '''Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Describe the structure of an atom.

### Response:
An atom is the basic building block of all matter and is made up of three types of particles: protons, neutrons, and electrons. The structure of an atom can be described as a nucleus at the center surrounded by a cloud of electrons.

The nucleus of an atom is made up of protons and neutrons. Protons are positively charged particles and neutrons are neutral particles with no charge. Both of these particles are located in the nucleus of the atom, which is at the center of the atom and contains most of the atom's mass.

Surrounding the nucleus of the atom is a cloud of electrons. Electrons are negatively charged particles that are in constant motion around the nucleus. The electron cloud is divided into shells or orbitals, and each shell can hold a certain number of electrons. The number of electrons in the outermost shell, called the valence shell, determines the chemical properties of the atom.

In a neutral atom, the number of protons in the nucleus is equal to the number of electrons in the electron cloud, so the positive and negative charges balance out and the atom has no overall charge. The number of protons, also called the atomic number, determines what element the atom is.'''
}

prompt = parse_alpaca_gpt4(row)
messages = [{ 'role': k, 'content': v } for k, v in prompt.items()]
print(generate_chat(messages))

In [ ]:
row = {
    'context': '''Komorida was born in Kumamoto Prefecture on July 10, 1981. After graduating from high school, he joined the J1 League club Avispa Fukuoka in 2000. Although he debuted as a midfielder in 2001, he did not play much and the club was relegated to the J2 League at the end of the 2001 season. In 2002, he moved to the J2 club Oita Trinita. He became a regular player as a defensive midfielder and the club won the championship in 2002 and was promoted in 2003. He played many matches until 2005. In September 2005, he moved to the J2 club Montedio Yamagata. In 2006, he moved to the J2 club Vissel Kobe. Although he became a regular player as a defensive midfielder, his gradually was played less during the summer. In 2007, he moved to the Japan Football League club Rosso Kumamoto (later Roasso Kumamoto) based in his local region. He played as a regular player and the club was promoted to J2 in 2008. Although he did not play as much, he still played in many matches. In 2010, he moved to Indonesia and joined Persela Lamongan. In July 2010, he returned to Japan and joined the J2 club Giravanz Kitakyushu. He played often as a defensive midfielder and center back until 2012 when he retired.''',
    'instruction': 'When was Tomoaki Komorida born?'
}

prompt = parse_databricks_dolly_15k(row)
messages = [{ 'role': k, 'content': v } for k, v in prompt.items()]
print(generate_chat(messages))

In [ ]:
row = {
    'conversations': [
      {
        "from": "human",
        "value": "As an online platform teacher named Aimee, you possess impeccable credentials which include a Bachelor of Science degree in Industrial and Labor Relations from Cornell University, expertise in the English language, and intermediate proficiency in both Chinese and Spanish. Additionally, your professional experience as a STEAM teacher at UN Women in Singapore has honed your skills in teaching children from the ages of 6-11 and working with students from all levels of education. Your exceptional teaching abilities in spoken English and pronunciation paired with your personal strengths of being informed, patient, and engaging make you an ideal teacher for students seeking to improve their English language skills. Can you provide a short, concise, and unique English self-introduction in bullet point form that would attract students to enroll in your course?"
      }
    ]
}

prompt = parse_wizardlm_evol_instruct_v2_196k(row)
messages = [{ 'role': k, 'content': v } for k, v in prompt.items()]
print(generate_chat(messages))

In [ ]:
row = {
    'conversations': [
      {
        "from": "system",
        "value": "You are an AI assistant. You will be given a task. You must generate a detailed and long answer.",
        "weight": None
      },
      {
        "from": "human",
        "value": "Produce a long descriptive sentence that uses all these words: Albuquerque, New Mexico, areaOfLand, 486.2 (square kilometres); Albuquerque, New Mexico, populationDensity, 1142.3 (inhabitants per square kilometre); Albuquerque, New Mexico, isPartOf, Bernalillo County, New Mexico; Albuquerque, New Mexico, areaTotal, 490.9 (square kilometres)",
        "weight": 0
      },
      {
        "from": "gpt",
        "value": "Stretching across a vast areaOfLand, totaling 486.2 square kilometres, Albuquerque, New Mexico, boasts a thriving populationDensity of approximately 1142.3 inhabitants per square kilometre, all residing within the expansive city limits which are part of the beautiful Bernalillo County in New Mexico, enveloping an impressive areaTotal of 490.9 square kilometres, encompassing both natural and urban landscapes.",
        "weight": 1
      }
    ]
}

prompt = parse_slimorca(row)
messages = [{ 'role': k, 'content': v } for k, v in prompt.items()]
print(generate_chat(messages))

In [ ]:
row = {
    'conversations':
      [
        {
          "from": "human",
          "value": "In analytical chemistry, what is the principle behind the use of an internal standard in quantitative analysis?\nA. It compensates for variations in sample preparation and instrumental response.\nB. It enhances the sensitivity of the analytical method.\nC. It reduces the detection limit of the analytical method.\nD. It increases the resolution between analyte peaks in chromatography.\nE. None of the above.",
          "weight": None
        },
        {
          "from": "gpt",
          "value": "A. It compensates for variations in sample preparation and instrumental response.",
          "weight": None
        }
      ]
}

prompt = parse_openhermes_25(row)
messages = [{ 'role': k, 'content': v } for k, v in prompt.items()]
print(generate_chat(messages))